# Spin-density matrix

:::{autolink-concat}
:::

The other {doc}`DPD notebooks </dpd>` formulate an **unpolarized** intensity: the spin projections of the initial state are summed over incoherently, which is what remains of the [differential decay rate](https://redeboer.github.io/phd-thesis/chapter3.html#sec-differential-decay-rate) after integrating over the orientation of the decay plane. This notebook formulates the **polarized** differential decay rate, which describes the distribution over the Dalitz variables _and_ the orientation of the decay plane. The orientation is parametrized by three Euler angles $(\phi, \theta, \chi)$ and the polarization of the initial state enters through a spin-density matrix $\rho$, a Hermitian matrix of dimension $(2j_0+1)$ whose diagonal elements give the populations of the spin projections $m_0$ along the quantization axis of the production frame and whose off-diagonal elements encode the coherences between them.

The differential decay rate can be written compactly as a matrix expression,

$$
I\left(\{\sigma\}; \phi, \theta, \chi\right) = \operatorname{tr}\left(
    \rho\,\overline{D}\,O\,\overline{D}^\dagger
\right),
$$

with $\overline{D}_{m_0,\nu} = D^{j_0*}_{m_0,\nu}(\phi, \theta, \chi)$ the matrix of conjugated Wigner-$D$ functions that rotate the initial state from the production frame to the aligned center-of-momentum frame and

$$
O_{\nu,\nu'}\left(\{\sigma\}\right) = \sum_{\{\lambda\}}
    O^\nu_{\{\lambda\}}\left(\{\sigma\}\right)\,
    O^{\nu'*}_{\{\lambda\}}\left(\{\sigma\}\right)
$$

the spin-density matrix of the decay itself, built from the aligned transition amplitudes $O^\nu_{\{\lambda\}}$ of the {doc}`Dalitz-plot decomposition </dpd>`. If $\rho = \mathbb{1}/(2j_0+1)$, the trace collapses to $\operatorname{tr}(O)/(2j_0+1)$ by unitarity of $\overline{D}$ and we recover the unpolarized intensity, independent of the orientation angles.

As an example, we use the decay $J/\psi \to K^0 \Sigma^+ \bar p$ with a small set of resonances. In $e^+e^-$ collisions, the $J/\psi$ is produced with only spin projections $m_0 = \pm 1$ along the beam axis, $\rho = \tfrac{1}{2}\operatorname{diag}(1, 0, 1)$, so the orientation of the decay plane carries additional information about the helicity couplings, even though the spin-density matrix introduces no new parameters.

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import qrules
import sympy as sp
from IPython.display import Latex

from ampform.adapter.qrules import normalize_state_ids, to_three_body_decay
from ampform.amplitude.dpd import DalitzPlotDecompositionBuilder
from ampform.dynamics.builder import formulate_breit_wigner_with_form_factor
from ampform.io import aslatex, improve_latex_rendering, mute_ampform_warnings
from ampform.sympy import cached

improve_latex_rendering()
warnings.simplefilter("ignore", category=RuntimeWarning)

if STATIC_PAGE := "EXECUTE_NB" in os.environ:
    mute_ampform_warnings()

## Decay definition

We keep only two intermediate resonances so that the symbolic expressions in this notebook remain small: a $N(1700)^+ \to K^0\Sigma^+$ resonance in subsystem&nbsp;3 and a $\bar\Sigma(1660)^- \to K^0\bar p$ resonance in subsystem&nbsp;2.

In [ ]:
REACTION = qrules.generate_transitions(
    initial_state="J/psi(1S)",
    final_state=["K0", "Sigma+", "p~"],
    allowed_interaction_types="strong",
    allowed_intermediate_particles=["N(1700)+", "Sigma(1660)"],
    formalism="canonical-helicity",
)
REACTION = normalize_state_ids(REACTION)
DECAY = to_three_body_decay(REACTION.transitions, min_ls=True)
Latex(aslatex(DECAY, with_jp=True))

## Model formulation

The polarized differential decay rate is formulated by passing {code}`polarized=True` to {meth}`.DalitzPlotDecompositionBuilder.formulate`. The intensity is now a trace over matrix symbols instead of an incoherent sum over the initial-state spin projections:

In [ ]:
model_builder = DalitzPlotDecompositionBuilder(DECAY, min_ls=True)
for chain in model_builder.decay.chains:
    model_builder.dynamics_choices.register_builder(
        chain, formulate_breit_wigner_with_form_factor
    )
model = model_builder.formulate(reference_subsystem=2, polarized=True)
model.intensity

The matrix symbols are defined as explicit matrices under {attr}`.AmplitudeModel.matrices`. Rows and columns are ordered by ascending spin projection, so for the $J/\psi$ the first row and column correspond to $m_0=-1$. Since $\rho$ and $O$ are Hermitian, only their upper triangles are formulated and the lower triangles are complex conjugates:

In [ ]:
Latex(aslatex(model.matrices))

The elements of the spin-density matrix $\rho$ enter the model as parameters. Their default values represent the unpolarized case, $\rho = \mathbb{1}/(2j_0+1)$:

In [ ]:
spin_density_parameters = {
    symbol: value
    for symbol, value in model.parameter_defaults.items()
    if str(symbol).startswith(R"\rho")
}
Latex(aslatex({k: sp.nsimplify(v) for k, v in spin_density_parameters.items()}))

The elements of the decay matrix $O$ are defined under {attr}`.AmplitudeModel.amplitudes`, in terms of the same subsystem amplitudes $A^k$ as in the unpolarized model (see {doc}`/dpd/jpsi2ksp`):

In [ ]:
decay_matrix_elements = {
    symbol: expression
    for symbol, expression in model.amplitudes.items()
    if str(symbol).startswith("O")
}
Latex(aslatex(decay_matrix_elements))

For comparison, the unpolarized model sums the same aligned amplitudes incoherently over all four helicities:

In [ ]:
unpolarized_model = model_builder.formulate(reference_subsystem=2)
unpolarized_model.intensity

## Preparing for input data

The decay matrix $O$ depends only on the Mandelstam variables, while the orientation angles $(\phi, \theta, \chi)$ enter only through $\overline{D}$. The Euler angles remain **free kinematic variables**: they are determined by the four-momenta of an event with respect to the production frame and cannot be expressed in terms of Mandelstam variables. We can therefore express each element of $O$ as a function over the Dalitz plane. We take $\sigma_2 = M^2(K^0\bar p)$ and $\sigma_3 = M^2(K^0\Sigma^+)$ as the independent Dalitz variables.

In [ ]:
σ1, σ1_expr = next(iter(model.invariants.items()))
kinematic_variables = {
    symbol: cached.doit(expression).xreplace({σ1: σ1_expr}).xreplace(model.masses)
    for symbol, expression in model.variables.items()
}


def to_dalitz_plane(expression: sp.Expr) -> sp.Expr:
    expression = cached.unfold(expression, model.amplitudes)
    expression = cached.xreplace(expression, model.parameter_defaults)
    expression = cached.xreplace(expression, kinematic_variables)
    expression = cached.xreplace(expression, {σ1: σ1_expr})
    return cached.xreplace(expression, model.masses)


dalitz_symbols = sp.symbols("sigma2 sigma3", nonnegative=True)
decay_matrix_funcs = {
    symbol.indices: sp.lambdify(dalitz_symbols, to_dalitz_plane(expression), "numpy")
    for symbol, expression in decay_matrix_elements.items()
}

In [ ]:
resolution = 500
m = sorted(model.masses, key=str)
x_min = float(((m[1] + m[3]) ** 2).xreplace(model.masses))
x_max = float(((m[0] - m[2]) ** 2).xreplace(model.masses))
y_min = float(((m[1] + m[2]) ** 2).xreplace(model.masses))
y_max = float(((m[0] - m[3]) ** 2).xreplace(model.masses))
x_diff = x_max - x_min
y_diff = y_max - y_min
x_min -= 0.05 * x_diff
x_max += 0.05 * x_diff
y_min -= 0.05 * y_diff
y_max += 0.05 * y_diff
X, Y = np.meshgrid(
    np.linspace(x_min, x_max, num=resolution),
    np.linspace(y_min, y_max, num=resolution),
)
spin_projections = [-1, 0, +1]
decay_matrix = np.zeros((3, 3, resolution, resolution), dtype=complex)
for i, ν in enumerate(spin_projections):
    for j, ν_prime in enumerate(spin_projections):
        if i <= j:
            decay_matrix[i, j] = decay_matrix_funcs[ν, ν_prime](X, Y)
        else:
            decay_matrix[i, j] = decay_matrix[j, i].conj()

## Dalitz plot

The unpolarized intensity is proportional to the trace of the decay matrix, $I_0 = \operatorname{tr}(O)$, which gives the familiar Dalitz distribution:

In [ ]:
%config InlineBackend.figure_formats = ['svg']

In [ ]:
dalitz_intensities = np.trace(decay_matrix).real
normalized_dalitz = dalitz_intensities / np.nansum(dalitz_intensities)

plt.rc("font", size=14)
fig, ax = plt.subplots(figsize=(9, 7))
mesh = ax.pcolormesh(X, Y, normalized_dalitz, rasterized=True)
ax.set_aspect("equal")
c_bar = plt.colorbar(mesh, ax=ax, pad=0.01)
c_bar.ax.set_ylabel("Normalized intensity (a.u.)")
ax.set_xlabel(R"$\sigma_2 = M^2\left(K^0 \bar{p}\right)$")
ax.set_ylabel(R"$\sigma_3 = M^2\left(K^0 \Sigma^+\right)$")
plt.show()

## Effect of polarization

The distribution over the orientation angles is obtained by integrating the decay matrix over the Dalitz plane, which leaves a numerical $3 \times 3$ matrix $O^\text{int}$, and substituting it into the trace expression. We compare two spin-density matrices:

1. **Unpolarized**: $\rho = \mathbb{1}/3$, the parameter defaults of the model.
2. **$e^+e^-$ production**: $\rho = \tfrac{1}{2}\operatorname{diag}(1, 0, 1)$, since the $J/\psi$ is produced through a virtual photon and hence carries no $m_0 = 0$ projection along the beam.

In [ ]:
integrated_decay_matrix = sp.ImmutableMatrix(
    np.nansum(decay_matrix, axis=(2, 3)).tolist()
)
ρ_matrix, D_matrix, O_matrix = model.matrices
orientation_expr = model.intensity.xreplace({
    ρ_matrix: model.matrices[ρ_matrix],
    D_matrix: model.matrices[D_matrix],
    O_matrix: integrated_decay_matrix,
}).doit()

ρ = sp.IndexedBase(R"\rho")
scenarios = {
    R"$\rho = \mathbb{1}/3$ (unpolarized)": dict(spin_density_parameters),
    R"$\rho = \frac{1}{2}\,\mathrm{diag}(1,0,1)$ ($e^+e^-$)": {
        **dict.fromkeys(spin_density_parameters, 0),
        ρ[-1, -1]: 0.5,
        ρ[+1, +1]: 0.5,
    },
}
euler_angles = sp.symbols("phi theta chi", real=True)
orientation_funcs = {
    title: sp.lambdify(euler_angles, orientation_expr.xreplace(parameters), "numpy")
    for title, parameters in scenarios.items()
}

Since both spin-density matrices are diagonal, the intensity does not depend on the first Euler angle $\phi$. In the unpolarized case, the distribution over the remaining angles $(\theta, \chi)$ is flat and equal to $\operatorname{tr}(O^\text{int})/3$, while the polarized spin-density matrix induces a clear modulation over the orientation of the decay plane:

In [ ]:
THETA, CHI = np.meshgrid(
    np.linspace(0, np.pi, num=200),
    np.linspace(-np.pi, +np.pi, num=200),
)
orientation_distributions = {
    title: np.broadcast_to(np.real(func(0, THETA, CHI)), THETA.shape)
    for title, func in orientation_funcs.items()
}
unpolarized_distribution = next(iter(orientation_distributions.values()))
np.testing.assert_allclose(
    3 * unpolarized_distribution,
    float(sp.re(integrated_decay_matrix.trace())),
    rtol=1e-10,
)

In [ ]:
normalization = float(sp.re(integrated_decay_matrix.trace())) / 3
v_max = max(
    np.max(distribution) / normalization
    for distribution in orientation_distributions.values()
)
fig, axes = plt.subplots(
    figsize=(12, 5), ncols=2, sharex=True, sharey=True, layout="constrained"
)
for ax, (title, distribution) in zip(
    axes, orientation_distributions.items(), strict=True
):
    mesh = ax.pcolormesh(
        CHI,
        np.cos(THETA),
        distribution / normalization,
        rasterized=True,
        vmin=0,
        vmax=v_max,
    )
    ax.set_title(title)
    ax.set_xlabel(R"$\chi$")
    ax.set_xticks([-np.pi, 0, +np.pi])
    ax.set_xticklabels([R"$-\pi$", "0", R"$+\pi$"])
axes[0].set_ylabel(R"$\cos\theta$")
c_bar = fig.colorbar(mesh, ax=axes, pad=0.02)
c_bar.ax.set_ylabel(R"$I(\theta, \chi)\,/\,I_\mathrm{unpolarized}$")
plt.show()

Projecting onto $\cos\theta$, the angle between the normal of the production plane and the quantization axis, shows the modulation that a fit to data gains as additional sensitivity:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5), layout="constrained")
theta = np.linspace(0, np.pi, num=300)
chi = np.linspace(-np.pi, +np.pi, num=300)
for title, func in orientation_funcs.items():
    values = func(0, theta[:, None], chi[None, :]).real
    values = np.broadcast_to(values, (len(theta), len(chi)))
    ax.plot(np.cos(theta), values.mean(axis=1) / normalization, label=title)
ax.set_xlabel(R"$\cos\theta$")
ax.set_ylabel(R"$I(\cos\theta)\,/\,I_\mathrm{unpolarized}$")
ax.set_ylim(bottom=0)
ax.legend()
plt.show()

Both distributions integrate to the same total intensity: polarization redistributes the events over the orientation of the decay plane, but does not change the total decay rate, nor the distribution over the Dalitz plane.

## Polarimeter decay of a final-state particle

Polarization information can also be extracted from the decay of one of the **final-state** particles. The weak decay $\Sigma^+ \to p\pi^0$ is self-analyzing: the proton is emitted preferentially along the $\Sigma^+$ spin, with a decay-asymmetry parameter $\alpha \approx -0.98$. Passing the decay products to {meth}`.DalitzPlotDecompositionBuilder.formulate` through {code}`final_state_decays` appends one Wigner-$D$ function and a helicity coupling per decay chain (see {func}`.attach_final_state_decays`),

$$
\widetilde{O}^\nu_{\lambda_1,\lambda_p,\lambda_{\pi^0},\lambda_3} = \sum_{\lambda_2}
    O^\nu_{\lambda_1,\lambda_2,\lambda_3}\,
    D^{1/2*}_{\lambda_2,\lambda_p}\!\left(\phi_p, \theta_p, 0\right)
    H^{\Sigma^+ \to p\pi^0}_{\lambda_p,\lambda_{\pi^0}}\,,
$$

where the helicity $\lambda_2$ of the $\Sigma^+$ is summed **coherently** and the summation over $\lambda_2$ in the intensity is replaced by **incoherent** summations over the helicities of the decay products. The angles $(\phi_p, \theta_p)$ are the direction of the proton in the aligned rest frame of the $\Sigma^+$ and enter the model as free kinematic variables:

In [ ]:
from ampform.decay import Particle

proton = Particle("p", latex="p", spin=0.5, parity=+1, mass=0.938, width=0.0)
pion = Particle("pi0", latex=R"\pi^0", spin=0, parity=-1, mass=0.135, width=0.0)
decayed_model = model_builder.formulate(
    reference_subsystem=2,
    final_state_decays={2: (proton, pion)},
)
decayed_model.intensity

The helicity couplings of the weak decay are not free parameters: they are fixed by the measured decay-asymmetry parameter through $H_{\pm 1/2,0} = \sqrt{(1\pm\alpha)/2}$, so the model gains sensitivity without gaining parameters.

In [ ]:
weak_couplings = {
    symbol: value
    for symbol, value in decayed_model.parameter_defaults.items()
    if str(symbol).startswith(R"\mathcal{H}^\mathrm{decay}[\Sigma^{+}")
}
α = -0.98
H_minus, H_plus = sorted(weak_couplings, key=str)
weak_couplings[H_plus] = np.sqrt((1 + α) / 2)
weak_couplings[H_minus] = np.sqrt((1 - α) / 2)
Latex(aslatex({k: v.round(4) for k, v in weak_couplings.items()}))

The payoff is that **relative phases** between helicity couplings, which can be invisible in the Dalitz distribution, become visible through the proton. The couplings $g_\pm = \mathcal{H}^\mathrm{decay}_{0,\pm 1/2}$ of the $N(1700)^+ \to K^0 \Sigma^+$ node determine the polarization of the $\Sigma^+$, but within the $N(1700)^+$ chain the Dalitz distribution only depends on $|g_+|^2 + |g_-|^2$: the relative phase $\delta = \arg(g_+ g_-^*)$ drops out. Below, we compare two models with $\delta = \pm\pi/2$ at a point where the two resonance bands cross. The Dalitz intensities are indistinguishable, but the phase generates a **transverse polarization** of the $\Sigma^+$ of opposite sign, which the polarimeter decay converts into opposite azimuthal modulations of the proton:

In [ ]:
g_plus = next(
    symbol
    for symbol in decayed_model.parameter_defaults
    if str(symbol) == R"\mathcal{H}^\mathrm{decay}[N(1700)^{+}, 0, 1/2]"
)
proton_angles = sp.symbols(R"\phi_{p} \theta_{p}", real=True)
dalitz_point = {
    sp.Symbol("sigma2", nonnegative=True): 2.76,
    sp.Symbol("sigma3", nonnegative=True): 2.96,
}


def substitute_kinematics(expression: sp.Expr) -> sp.Expr:
    expression = cached.xreplace(expression, kinematic_variables)
    expression = cached.xreplace(expression, {σ1: σ1_expr})
    expression = cached.xreplace(expression, model.masses)
    return expression.xreplace(dalitz_point)


def create_proton_distribution(δ: float):
    parameters = dict(decayed_model.parameter_defaults)
    parameters.update(weak_couplings)
    parameters[g_plus] = np.exp(1j * δ)
    expression = cached.unfold(decayed_model)
    expression = cached.xreplace(expression, parameters)
    return sp.lambdify(proton_angles, substitute_kinematics(expression), "numpy")


def compute_dalitz_intensity(δ: float) -> float:
    parameters = dict(unpolarized_model.parameter_defaults)
    parameters[g_plus] = np.exp(1j * δ)
    expression = cached.unfold(unpolarized_model)
    expression = cached.xreplace(expression, parameters)
    return complex(substitute_kinematics(expression)).real


proton_distributions = {
    Rf"$\delta = {sign}\pi/2$": create_proton_distribution(δ)
    for sign, δ in {"+": +np.pi / 2, "-": -np.pi / 2}.items()
}
dalitz_intensities = [compute_dalitz_intensity(δ) for δ in (+np.pi / 2, -np.pi / 2)]
np.testing.assert_allclose(*dalitz_intensities, rtol=1e-3)

Without the polarimeter decay, the two models are essentially indistinguishable at this point on the Dalitz plane, as the assertion above shows. The proton distributions, however, are modulated in opposite directions:

In [ ]:
PHI_P, COS_THETA_P = np.meshgrid(
    np.linspace(-np.pi, +np.pi, num=200, endpoint=False),
    np.linspace(-1, +1, num=201),
)
proton_intensities = {
    title: np.real(func(PHI_P, np.arccos(COS_THETA_P)))
    for title, func in proton_distributions.items()
}
integrals = [
    2 * np.pi * np.trapezoid(intensity.mean(axis=1), x=COS_THETA_P[:, 0])
    for intensity in proton_intensities.values()
]
expected = [2 * np.pi * intensity for intensity in dalitz_intensities]
np.testing.assert_allclose(integrals, expected, rtol=1e-6)

In [ ]:
mean_intensity = np.mean(list(proton_intensities.values()))
v_diff = max(
    np.max(np.abs(intensity / mean_intensity - 1))
    for intensity in proton_intensities.values()
)
fig, axes = plt.subplots(
    figsize=(12, 5), ncols=2, sharex=True, sharey=True, layout="constrained"
)
for ax, (title, intensity) in zip(axes, proton_intensities.items(), strict=True):
    mesh = ax.pcolormesh(
        PHI_P,
        COS_THETA_P,
        intensity / mean_intensity,
        cmap="RdBu_r",
        rasterized=True,
        vmin=1 - v_diff,
        vmax=1 + v_diff,
    )
    ax.set_title(title)
    ax.set_xlabel(R"$\phi_p$")
    ax.set_xticks([-np.pi, 0, +np.pi])
    ax.set_xticklabels([R"$-\pi$", "0", R"$+\pi$"])
axes[0].set_ylabel(R"$\cos\theta_p$")
c_bar = fig.colorbar(mesh, ax=axes, pad=0.02)
c_bar.ax.set_ylabel(R"$I(\phi_p, \theta_p)\,/\,\bar{I}$")
plt.show()

Integrated over the proton angles, both models collapse to the same value $2\pi I_0$, so this information is invisible without the polarimeter decay. Final-state decays can also be combined with {code}`polarized=True` to formulate a differential decay rate that is sensitive to the initial-state spin-density matrix _and_ the polarization of a final-state particle simultaneously.